# 6.1 Code Brief: Additional Boosting Algorithms

Condensed reference for notebook 6.1.

## Comparison of All Boosting Methods

| Feature | AdaBoost | XGBoost | LightGBM | CatBoost |
|:--------|:---------|:--------|:---------|:---------|
| **Year** | 1996 | 2014 | 2017 | 2017 |
| **Strategy** | Sample reweighting | Gradient on residuals | Gradient + GOSS | Ordered boosting |
| **Speed** | Moderate | Fast | Very fast | Fast |
| **Categorical handling** | Needs encoding | Needs encoding | Native (integer) | Native (string) |
| **Missing data** | Needs imputation | Native | Native | Native |
| **Best for** | Historical interest | General purpose | Large data, speed | Categorical features |
| **In this course** | Special topic | Core model (Module 3) | Special topic | Special topic |

## Setup and Data Preparation

In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

# Load data (same preparation as Module 2)
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
train_df = pd.read_csv(f'{filepath}training.csv')
test_df = pd.read_csv(f'{filepath}testing.csv')
train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',
    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']

train_enc = pd.get_dummies(train_df[numeric_features + categorical_features],
                           columns=categorical_features, drop_first=True)
test_enc = pd.get_dummies(test_df[numeric_features + categorical_features],
                          columns=categorical_features, drop_first=True)
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)

# Impute with TRAIN medians only, never test's own (avoids leakage)
train_medians = train_enc.median()
train_enc = train_enc.fillna(train_medians)
test_enc = test_enc.fillna(train_medians)

X_train, y_train = train_enc, train_df['DEPARTED']
X_test, y_test = test_enc, test_df['DEPARTED']

## AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
# AdaBoost
ada = AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42)
ada.fit(X_train, y_train)
ada_prob = ada.predict_proba(X_test)[:, 1]

print(f"AdaBoost ROC-AUC: {roc_auc_score(y_test, ada_prob):.4f}")

## LightGBM

In [ ]:
try:
    from lightgbm import LGBMClassifier

    lgbm = LGBMClassifier(
        n_estimators=150, learning_rate=0.1, max_depth=5,
        num_leaves=31, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=42, verbose=-1
    )
    lgbm.fit(X_train, y_train)
    lgbm_prob = lgbm.predict_proba(X_test)[:, 1]

    print(f"LightGBM ROC-AUC: {roc_auc_score(y_test, lgbm_prob):.4f}")

except ImportError:
    print("LightGBM not installed. Install with: pip install lightgbm")

## CatBoost

In [ ]:
!pip install catboost

In [ ]:
try:
    from catboost import CatBoostClassifier

    cat = CatBoostClassifier(
        iterations=150, learning_rate=0.1, depth=5,
        auto_class_weights='Balanced',
        random_seed=42, verbose=0
    )
    cat.fit(X_train, y_train)
    cat_prob = cat.predict_proba(X_test)[:, 1]

    print(f"CatBoost ROC-AUC: {roc_auc_score(y_test, cat_prob):.4f}")

except ImportError:
    print("CatBoost not installed. Install with: pip install catboost")

## Key Takeaways
- **AdaBoost**: Historical importance, largely superseded by gradient boosting
- **LightGBM**: Choose for very large datasets or when speed is critical
- **CatBoost**: Choose when you have many categorical features and want easy setup
- **For this course**: XGBoost is the recommended boosting algorithm for practical use

**Next:** 6.2 Special Topics: Neural Networks